# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# The dataset.metadata object provides summary info
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets in the dataset, by their '@id' and 'name'
print("Available record sets:")
record_set_ids = []
for rs in dataset.metadata.record_sets:
    print(f"  @id: {rs.id}    | name: {rs.name}")
    record_set_ids.append(rs.id)

# Inspect fields/columns for each record set
for rs in dataset.metadata.record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    print("Fields/Columns:")
    for f in rs.fields:
        print(f"  - {f.id}   (name: {f.name}, type: {getattr(f, 'data_type', None)})")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Below, we use record set and field `@id`s from the overview above.

For this dataset, we'll load the primary protein records table.

In [ ]:
# List of record_set @id's (from overview)
record_sets = record_set_ids  # Use those detected above
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  (No records found for {record_set_id})")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame with shape: {df.shape}")
    print(f"Columns for {record_set_id}:")
    print(df.columns.tolist())

# Pick the main record set for further analysis
main_rs_id = None
if len(dataframes) > 0:
    main_rs_id = list(dataframes)[0]
    print(f"\nPreview of first few rows for record set {main_rs_id}:")
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes extracted. Please check the record sets returned by croissant metadata.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# Pick a numeric field for demonstration
# Here, attempt to automatically use a column like 'coverage_percent' or 'MW' (molecular weight), as example field names

import numpy as np

df = None
if main_rs_id:
    df = dataframes[main_rs_id]
if df is None or df.empty:
    print("No data available for EDA.")
else:
    # Try to pick likely numeric field
    candidate_numeric = None
    for col in df.columns:
        if any(x in col.lower() for x in ['mw', 'molecular_weight', 'coverage', 'abundance', 'count', 'number', 'peptide', 'area']):
            candidate_numeric = col
            if np.issubdtype(df[col].dtype, np.number):
                break
    # If not already numeric, try converting
    if candidate_numeric is not None and not np.issubdtype(df[candidate_numeric].dtype, np.number):
        try:
            df[candidate_numeric] = pd.to_numeric(df[candidate_numeric], errors='coerce')
        except Exception:
            pass

    if candidate_numeric is None:
        print('No obvious numeric field found!')
    else:
        numeric_field = candidate_numeric
        print(f"Analyzing numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
        print(filtered_df[[numeric_field]].head())

        # Normalize field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field (e.g. 'description' or others)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and candidate_numeric is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[candidate_numeric].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {candidate_numeric}")
    plt.xlabel(candidate_numeric)
    plt.ylabel("Frequency")
    plt.show()

    # If a second numeric field exists, show scatterplot
    num_cols = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number) and col != candidate_numeric]
    if len(num_cols) > 0:
        b = num_cols[0]
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[candidate_numeric], y=df[b])
        plt.xlabel(candidate_numeric)
        plt.ylabel(b)
        plt.title(f"Scatter: {candidate_numeric} vs {b}")
        plt.show()

## 6. Conclusion

This notebook demonstrated how to use the `mlcroissant` library to load, inspect, and perform basic analysis on the FAIR^2 dataset representing mass spectrometry data of extracellular vesicle proteins from human mast cells.

- We listed available record sets and fields using their `@id`s for programmatic reference.
- Extracted and visualized tabular protein data by field.
- Applied sample filtering, normalization, and summary statistics for exploratory analysis.

Further work could include deeper domain-specific analysis, feature engineering, or integrating this data with other proteomics/omics measurements.